In [ ]:
import msprime, tskit
import numpy as np
import gaiapy as gp
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from tqdm import tqdm
import time

In [ ]:
def find_children(unary_indices, node_stats_df):
    start = time.time()
    children = set()
    for node_id in unary_indices:
        n = node_stats_df.loc[node_id, 'distinct_children']
        if n is None or len(n) == 0:
            continue
        n = n[n != -1]
        children.update(n)
    end = time.time()
    runtime = end - start
    return np.array(list(children)), runtime

In [ ]:
#here ts is the unsimplified tree
def get_node_stats(ts):
    #nodes = list(ts.nodes())
    nodes = np.array(list(ts.nodes()))
    node_ids = np.array([n.id for n in nodes])
    # Find the samples
    #is_sample = np.asarray(np.isin(nodes, ts.samples()), dtype=int)
    is_sample = np.asarray(np.isin(node_ids, ts.samples()), dtype=int)
    # Find all the other things (this requires checking tree by tree)
    tree = ts.first()
    start = tree.interval[0] 
    end = ts.sequence_length
    num_children = np.zeros(nodes.shape[0])
    num_parents = np.zeros(nodes.shape[0])
    distinct_children, distinct_parents, distinct_populations = list(), list(), list()
    is_root = np.zeros(nodes.shape[0])
    #for i,node in tqdm(enumerate(nodes)):
    for i, node_id in tqdm(enumerate(node_ids)): 
        tree.seek(start)
        children = list()
        parents = list()
        is_root[i] = tree.is_root(node_id)
        node_children = list(tree.children(node_id))

        children.extend(node_children)
        parents.append(tree.parent(node_id))

        w = (None, tree.interval[0])
        while w[1] < end and tree.next():
            w = (w[1], min(tree.interval[1], end))
            is_root[i] = tree.is_root(node_id)
            node_children = list(tree.children(node_id))

            children.extend(node_children)
            parents.append(tree.parent(node_id))

        distinct_parents.append(np.unique(parents))
        distinct_children.append(np.unique(children))
        num_children[i] = np.unique(children).shape[0]
        num_parents[i] = np.unique(parents).shape[0]
    
    data_dict = {
        'id': node_ids,
        'num_children': num_children,
        'distinct_children': distinct_children,
        'distinct_parents': distinct_parents,
        'num_parents': num_parents,
        'is_sample': is_sample,
        'is_root': is_root,
    }
    return pd.DataFrame(data_dict)

In [ ]:

#function that takes the tree sequence and finds the unary nodes in the tree sequence
# returns a list of all the unary indices by node id 
def findUnary(ts):
    unary_nodes = np.zeros(ts.num_nodes) # binary vector specifying if a node is unary or not anywhere on the tree sequence
    for tree in ts.trees():
        num_children = tree.num_children_array[:ts.num_nodes]
        is_unary = num_children == 1
        for i, condition in enumerate(is_unary):
            if is_unary[i] == True:
                unary_nodes[i] = 1
    mask = unary_nodes == 1
    #mask, unary_nodes[mask]
    unary_list = np.where(mask)
    unary_indices = unary_list[0]
    return unary_indices

In [ ]:
def print_children(node_stats_df):
    for n in node_stats_df['distinct_children']:
        print(n)
    return


In [ ]:
def print_parents(node_stats_df):
    for n in node_stats_df['distinct_parents']:
        print(n)
    return

In [ ]:
ts1 = msprime.sim_ancestry(
    samples=100,            # 6 diploid individuals = 12 haploid chromosomes
    population_size=1000,
    sequence_length=10_000,
    recombination_rate=1e-6,   # low recomb → likely a single tree
    random_seed=42,
    record_full_arg=True,
    #coalescing_segments_only=False,
    record_provenance=True
)

ts1_unary = findUnary(ts1)
ts1_node_df = get_node_stats(ts1)
ts1_node_df.head()
find_children(ts1_unary, ts1_node_df)


In [ ]:
#ts2 = msprime.sim_ancestry(1e5, sequence_length=1e8, recombination_rate=1e-8, population_size=1e5, coalescing_segments_only=False, random_seed=1)

In [ ]:
ts3 = msprime.sim_ancestry(3, recombination_rate=0.2, sequence_length=3, record_full_arg=True, random_seed=42, record_provenance=True)
# Example: store age and location for each individual
tables = ts3.dump_tables()

node_schema = tskit.MetadataSchema({
    "codec": "json",
    "type": "object",
    "properties": {
        "slim_id": {"type": "integer"}
    }
})
tables.nodes.metadata_schema = node_schema


for node_id, node in enumerate(tables.nodes):
    tables.nodes[node_id] = node.replace(metadata={"slim_id": node_id})


ts3 = tables.tree_sequence()

# tested.tables.nodes

sts3 = ts3.simplify(filter_nodes=False)
ets3 = sts3.extend_haplotypes()

In [ ]:
ts3_unary = findUnary(ts3)
ts3_node_df = get_node_stats(ts3)
for n in ts3_unary:
    print(n)



In [ ]:
# Display all columns
pd.set_option('display.max_columns', None)

# Display all rows
pd.set_option('display.max_rows', None)

# Display the DataFrame
display(ts3_node_df)